In [1]:
#HouseKeeping
import numpy as np 
import pandas as pd 

data = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')

In [2]:
#BASIC STUFF

data = np.array(data)
np.random.shuffle(data)
results = 10 #number of nodes output

test_data = data[0:1000].T
test_y  = test_data[0]
test_x  = test_data[1:]/255

train_data = data[1000:].T
train_y = train_data[0]
train_x = train_data[1:]/255

pixels, train_trials = train_x.shape

train_y.size

41000

In [3]:
def init_parameters(H, N, pixels, results):
    """
    Initialize weights and biases for a neural network with H hidden layers.

    Parameters
    ----------
    H : int
        Number of hidden layers.
    N : int
        Number of neurons in each hidden layer.
    pixels : int
        Input dimension (e.g., flattened image pixels).
    results : int
        Output dimension (number of classes).

    Returns
    -------
    list of lists
        A list where each element is a layer's parameters [W, b]:
        - W: Weight matrix (shape [neurons_current, neurons_previous]).
        - b: Bias vector (shape [neurons_current, 1]).
    """
    params = []
    # 1st layer
    params.append([
        np.random.rand(N, pixels) - 0.5,  
        np.random.rand(N, 1) - 0.5       
    ])
    # the rest
    for _ in range(H-1):
        params.append([
            np.random.rand(N, N) - 0.5,   
            np.random.rand(N, 1) - 0.5    
        ])
    # output layer
    params.append([
        np.random.rand(results, N) - 0.5,  
        np.random.rand(results, 1) - 0.5   
    ])
    return params


def ReLu(Z):
    """
    Rectified Linear Unit (ReLU) activation function.

    Parameters
    ----------
    Z : numpy.ndarray
        Pre-activation input matrix.

    Returns
    -------
    numpy.ndarray
        Element-wise maximum of 0 and Z.
    """
    return np.maximum(0,Z)
def softmax(Z): 
    """
    Rectified Linear Unit (ReLU) activation function.

    Parameters
    ----------
    Z : numpy.ndarray
        Pre-activation input matrix.

    Returns
    -------
    numpy.ndarray
        Element-wise maximum of 0 and Z.
    """
    return np.exp(Z)/sum(np.exp(Z))

def forward_propogation(IN, params):
    """
    Perform forward propagation through the network.

    Parameters
    ----------
    IN : numpy.ndarray
        Input data (shape [input_dim, batch_size]).
    params : list of lists
        Network parameters (weights and biases per layer).

    Returns
    -------
    tuple
        - OUT: Output probabilities (shape [output_dim, batch_size]).
        - Zs: List of pre-activations for each layer.
        - As: List of activations (including input layer).
    """
    Zs, As = [], [IN]  # As starts with input X
    
    for W, b in params[:-1]:  # Process all but last layer
        Z = np.dot(W, As[-1]) + b
        A = ReLu(Z)  # or sigmoid for first layer
        Zs.append(Z)
        As.append(A)
    
    # Output layer (softmax)
    W_out, b_out = params[-1]
    Z_out = np.dot(W_out, As[-1]) + b_out
    OUT = softmax(Z_out)
    
    return OUT, Zs, As
        

In [4]:
def one_hot(Y):
    """
    Convert class labels to one-hot encoded vectors.

    Parameters
    ----------
    Y : numpy.ndarray
        Class labels (shape [batch_size,]).

    Returns
    -------
    numpy.ndarray
        One-hot encoded matrix (shape [num_classes, batch_size]).
    """
    Y_one_hot = np.zeros((Y.size, Y.max()+1)) #LEARN NEED (())
    Y_one_hot[np.arange(Y.size),Y] = 1    #LEARN
    return Y_one_hot.T

def derv_sigmoid(Z):
    return np.exp(-Z)/(1+np.exp(-Z))**2
def derv_ReLu(Z): 
    """
    Derivative of the ReLU function.

    Parameters
    ----------
    Z : numpy.ndarray
        Pre-activation input matrix.

    Returns
    -------
    numpy.ndarray
        Binary mask where 1 indicates Z > 0, else 0.
    """
    return Z > 0

def back_propogation(OUT, train_y, Z, A, params):
    """
    Compute gradients for all parameters using backpropagation.

    Parameters
    ----------
    OUT : numpy.ndarray
        Output probabilities from forward pass.
    train_y : numpy.ndarray
        Ground truth labels (shape [batch_size,]).
    Z : list of numpy.ndarray
        Pre-activations from forward pass.
    A : list of numpy.ndarray
        Activations from forward pass (including input).
    params : list of lists
        Current network parameters.

    Returns
    -------
    list of lists
        Gradients for each layer [dW, db], ordered from output to input.
    """
    Y = one_hot(train_y)
    m = train_y.size

    gradient = []

    dz = OUT - Y
    dw = 1/m * np.dot(dz, A[-2].T) 
    db = 1/m * np.sum(dz, axis=1, keepdims=True)
    gradient.append([dw, db])
    
    for i in range(1, len(params)):
        '
        
        ms'
        dw = 1/m * np.dot(dz, A[-i-1].T) 
        db = 1/m * np.sum(dz, axis=1, keepdims=True)
        gradient.append([dw, db])
    
    return gradient[::-1]  # Reverse to match params order

In [7]:
def get_accuracy(A,Y):
    """
    Calculate classification accuracy.

    Parameters
    ----------
    A : numpy.ndarray
        Predicted probabilities (shape [num_classes, batch_size]).
    Y : numpy.ndarray
        True labels (shape [batch_size,]).

    Returns
    -------
    float
        Accuracy between 0 and 1.
    """
    return sum(np.argmax(A,0) == Y)/Y.size #LEARN ARGMAX

def gradient_descent(H, N, X, Y, pixels, results, iterations, alpha):
    """
    Train the neural network using gradient descent.

    Parameters
    ----------
    H : int
        Number of hidden layers.
    N : int
        Neurons per hidden layer.
    X : numpy.ndarray
        Training data (shape [input_dim, batch_size]).
    Y : numpy.ndarray
        Training labels (shape [batch_size,]).
    pixels : int
        Input dimension.
    results : int
        Output dimension (number of classes).
    iterations : int
        Number of training iterations.
    alpha : float
        Learning rate.

    Returns
    -------
    list of lists
        Trained parameters for each layer.
    """
    params = init_parameters(H, N, pixels, results)
    for i in range(iterations):
        OUT, Z, A = forward_propogation(X, params)
        gradient = back_propogation(OUT, Y, Z, A, params)
        
        # Update parameters
        for j in range(len(params)):
            params[j][0] -= alpha * gradient[-j][0]
            params[j][1] -= alpha * gradient[-j][1]
        
        if i % 100 == 0:
            alpha = alpha
            print(f"Iteration:{i}")
            print(get_accuracy(A[-1], Y))

    return params

In [8]:
params = init_parameters(3,10,pixels,results)
OUT,Z,A = forward_propogation(train_x,params)
gradient = back_propogation(OUT,train_y,Z,A,params)

In [8]:
params = gradient_descent(3,10,train_x,train_y,pixels,results,1000,0.1)

Iteration:0
0.11609756097560976
Iteration:100
0.1305609756097561
Iteration:200
0.12078048780487805
Iteration:300
0.0933170731707317
Iteration:400
0.07514634146341463
Iteration:500
0.1503170731707317
Iteration:600
0.13026829268292683
Iteration:700
0.15048780487804878
Iteration:800
0.14778048780487804
Iteration:900
0.1784390243902439
